# Example: Rebalancing Engine and Scorecard

In this example, we wire the **Cobb-Douglas utility allocator** into a full rebalancing engine with trigger rules (drawdown limits, turnover caps), produce a scorecard comparing the engine to baseline strategies, and run a sensitivity analysis over the key parameters $\lambda$ (risk-aversion) and $\epsilon$ (minimum share floor).

> **By the end of this example, you will be able to:**
> * __Run the Cobb-Douglas rebalancing engine:__ Execute the rebalancing engine with production-style trigger rules including drawdown limits and turnover caps. Observe how tighter or looser drawdown thresholds affect capital protection and recovery.
> * __Produce a strategy scorecard:__ Build a scorecard comparing the AI engine to buy-and-hold and risk-free benchmarks. Evaluate return, volatility, Sharpe ratio, and maximum drawdown across strategies.
> * __Analyze sensitivity to key parameters:__ Sweep the lambda gain and epsilon parameters to map their effect on portfolio performance. Identify the parameter sweet spot that balances responsiveness with diversification.

Let's dive in!

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

Load the data from Example 1, including market data, engine context, and benchmark wealth series. The `let...end` block below loads price data into the `price_matrix::Matrix{Float64}` variable, the engine context into the `context::MyRebalancingContextModel` variable, and the benchmark wealth series into `buyhold_wealth::Array{Float64,1}` and `riskfree_wealth` variables.

In [ ]:
let
    # --- Step 1: Load saved data from Example 1 ---
    data = load(joinpath(_PATH_TO_DATA, "engine-run-data.jld2"));

    # --- Step 2: Extract market and engine data into global scope ---
    global price_matrix = data["price_matrix"];       # (T × K) matrix of asset prices
    global lambda_series = data["lambda_series"];      # time-varying risk-aversion signal
    global market_prices = data["market_prices"];      # market factor price series
    global gm_ema = data["gm_ema"];                    # EMA-smoothed market growth factor
    global tickers = data["tickers"];                  # vector of ticker symbols
    global sim_params = data["sim_params"];             # simulation parameter struct
    global context = data["context"];                   # rebalancing engine context model
    global buyhold_wealth = data["buyhold_wealth"];     # buy-and-hold benchmark wealth
    global riskfree_wealth = data["riskfree_wealth"];   # risk-free benchmark wealth

    # --- Step 3: Define trading calendar constants ---
    global Δt = 1.0 / 252.0;          # daily time step (fraction of a trading year)
    global N_short = 21;               # short EMA window (approx 1 month)
    global N_long = 63;                # long EMA window (approx 3 months)
    global offset = N_short + N_long;  # warmup period before trading starts
    global n_trading_days = 252;       # number of trading days in the simulation
    global B₀ = context.B;            # initial budget from the context
    global K = length(tickers);        # number of assets in the portfolio

    println("Loaded engine data: $(K) tickers, $(n_trading_days) trading days")
end

___
## Task 1: Cobb-Douglas Rebalancing Engine with Trigger Rules
We run the Cobb-Douglas utility allocator inside the full rebalancing engine with realistic trigger rules: a 15% drawdown limit (circuit breaker) and a 50% turnover cap (trading cost control). We compare three drawdown thresholds to see how the safety net affects performance.

> __What should you see?__
>
> Tighter drawdown limits (10%) cause the engine to de-risk to cash earlier and more often, protecting capital but potentially missing recoveries. Looser limits (25%) allow more volatility. The engine uses Cobb-Douglas utility to decide _how_ to allocate; the trigger rules decide _whether_ to allocate.

The code below sweeps three drawdown limits using [the `run_rebalancing_engine(...)` function](../../code/docs/build/session2.html), and the results are stored in the `dd_wealth_curves::Dict{Float64, Array{Float64,1}}` variable.

In [ ]:
let
    # --- Step 1: Define drawdown thresholds and plot styling ---
    drawdown_limits = [0.10, 0.15, 0.25];   # three drawdown trigger levels to compare
    colors = [:red :orange :green];           # color per threshold
    labels = ["DD ≤ 10%", "DD ≤ 15%", "DD ≤ 25%"];  # legend labels

    # Storage: maps each drawdown limit to its wealth curve
    global dd_wealth_curves = Dict{Float64, Array{Float64,1}}();

    # --- Step 2: Initialize the plot ---
    p = plot(size=(800, 450), title="Cobb-Douglas Engine: Drawdown-Triggered De-Risking",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)", legend=:topleft)

    # --- Step 3: Sweep drawdown limits ---
    for (j, dd_limit) in enumerate(drawdown_limits)

        # Build trigger rules: drawdown limit varies, turnover cap fixed at 50%
        rules = build(MyTriggerRules, (
            max_drawdown = dd_limit,
            max_turnover = 0.50,
            rebalance_schedule = ones(Int, n_trading_days)  # rebalance every day
        ));

        # Run the rebalancing engine with Cobb-Douglas allocator
        results = run_rebalancing_engine(context, rules, lambda_series;
            offset=offset, allocator=:cobb_douglas);

        # Convert allocation results into a wealth time series
        wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);
        dd_wealth_curves[dd_limit] = wealth;  # store for later use

        # Plot this drawdown scenario
        plot!(p, 1:length(wealth), wealth, label=labels[j], linewidth=2, color=colors[j])
    end

    # --- Step 4: Add buy-and-hold reference line ---
    plot!(p, 1:length(buyhold_wealth), buyhold_wealth, label="Buy-and-Hold",
        linewidth=1.5, color=:gray50, linestyle=:dash)
    p
end

___
## Task 2: Scorecard, Engine vs. Baselines
We produce a scorecard comparing three strategies: the AI Cobb-Douglas engine (DD $\leq$ 15%, $\tau \leq$ 50%), equal-weight buy-and-hold, and risk-free. The scorecard tracks return, volatility, Sharpe ratio, and maximum drawdown.

> __What should you see?__
>
> The engine should show better risk-adjusted performance (lower drawdown, potentially better Sharpe) than static allocations, at the cost of higher trading activity. The scorecard quantifies exactly how much adaptability costs and what it buys.

The code below runs the engine using [the `run_rebalancing_engine(...)` function](../../code/docs/build/session2.html), computes per-strategy metrics, and stores the results in a `scorecard::DataFrame` variable.

In [ ]:
let
    # --- Step 1: Run the moderate engine configuration (DD ≤ 15%, τ ≤ 50%) ---
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series;
        offset=offset, allocator=:cobb_douglas);
    global engine_wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

    # --- Step 2: Define a helper to compute performance metrics from a wealth series ---
    function scorecard_metrics(wealth::Array{Float64,1}, label::String)
        returns = diff(wealth) ./ wealth[1:end-1];          # daily simple returns
        total_return = (wealth[end] / wealth[1] - 1.0) * 100;  # cumulative return (%)
        vol = std(returns) * sqrt(252) * 100;                # annualized volatility (%)
        sharpe = vol > 0 ? total_return / vol : 0.0;         # Sharpe ratio (return/vol)
        peak = accumulate(max, wealth);                       # running peak wealth
        dd = maximum((peak .- wealth) ./ peak) * 100;        # max drawdown (%)
        return (label, round(total_return, digits=2), round(vol, digits=2),
            round(sharpe, digits=2), round(dd, digits=2))
    end

    # --- Step 3: Compute metrics for each strategy ---
    eng = scorecard_metrics(engine_wealth, "Cobb-Douglas Engine (DD≤15%, τ≤50%)");
    bh = scorecard_metrics(buyhold_wealth, "Buy-and-Hold (equal-weight)");
    rf = scorecard_metrics(collect(riskfree_wealth), "Risk-Free (4.3%)");

    # --- Step 4: Build and display the scorecard table ---
    scorecard = DataFrame(
        "Strategy" => [eng[1], bh[1], rf[1]],
        "Return (%)" => [eng[2], bh[2], rf[2]],
        "Volatility (%)" => [eng[3], bh[3], rf[3]],
        "Sharpe Ratio" => [eng[4], bh[4], rf[4]],
        "Max Drawdown (%)" => [eng[5], bh[5], rf[5]]
    );

    println("═"^70)
    println("  SESSION 2 SCORECARD: Cobb-Douglas Engine vs. Baselines")
    println("═"^70)
    pretty_table(scorecard, tf=tf_markdown)

    # --- Step 5: Save results for Session 3 consumption ---
    save(joinpath(_PATH_TO_DATA, "session2-scorecard.jld2"),
        Dict("scorecard" => scorecard, "engine_wealth" => engine_wealth,
             "buyhold_wealth" => buyhold_wealth));
end

The code below plots the wealth curves for all three strategies (engine, buy-and-hold, risk-free) on a single axis.

In [ ]:
let
    # --- Step 1: Define the x-axis (trading days after warmup) ---
    days = 1:length(engine_wealth);

    # --- Step 2: Plot each strategy ---
    plot(days, engine_wealth, label="Cobb-Douglas Engine", linewidth=2.5, color=:steelblue)
    plot!(days, buyhold_wealth, label="Buy-and-Hold", linewidth=2, color=:coral, linestyle=:dash)
    plot!(days, collect(riskfree_wealth), label="Risk-Free (4.3%)", linewidth=1.5, color=:gray50, linestyle=:dot)

    # --- Step 3: Add axis labels and formatting ---
    xlabel!("Trading Day (after warmup)")
    ylabel!("Wealth (\$)")
    title!("Session 2: Cobb-Douglas Rebalancing Engine vs. Baselines")
    plot!(size=(800, 450), legend=:topleft)
end

___
## Task 3: Sensitivity Analysis, Lambda Gain and Epsilon
We sweep two key parameters to understand how they affect portfolio performance:

1. **Lambda gain $G$** controls how aggressively the sentiment signal modulates risk. Higher $G$ means the engine reacts more strongly to EMA crossover.
2. **Epsilon $\epsilon$** is the minimum share floor for non-preferred assets. Higher $\epsilon$ means more capital is locked in non-preferred positions (more diversified but less responsive).

> __What should you see?__
>
> A heatmap showing how Sharpe ratio (or final wealth) varies across the $(G, \epsilon)$ parameter space. There should be a sweet spot: too low $G$ makes the engine unresponsive, while too high $G$ makes it over-reactive. Too low $\epsilon$ concentrates risk, while too high $\epsilon$ wastes budget on non-preferred assets.

The code below sweeps a grid of $(G, \epsilon)$ values, running [the `run_rebalancing_engine(...)` function](../../code/docs/build/session2.html) at each point. The Sharpe ratios and final wealth values are stored in the `sharpe_grid::Matrix{Float64}` and `wealth_grid::Matrix{Float64}` variables.

In [ ]:
let
    # --- Step 1: Define parameter grids ---
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];   # lambda gain values to sweep
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];     # min share floor values to sweep

    # --- Step 2: Allocate storage for results ---
    global sharpe_grid = zeros(length(G_values), length(ε_values));  # Sharpe ratio at each (G, ε)
    global wealth_grid = zeros(length(G_values), length(ε_values));  # final wealth at each (G, ε)

    # --- Step 3: Sweep over all (G, ε) combinations ---
    for (i, G) in enumerate(G_values)
        for (j, ε) in enumerate(ε_values)

            # Recompute the lambda series with this gain G
            ema_s = compute_ema(market_prices; window=21);   # short-term EMA
            ema_l = compute_ema(market_prices; window=63);   # long-term EMA
            λ_new = compute_lambda(ema_s, ema_l; G=G);      # lambda = G * (EMA_short - EMA_long)
            λ_new[1:offset] .= 0.0;                          # zero out warmup period

            # Build a new context with this epsilon value
            ctx = build(MyRebalancingContextModel, (
                B = B₀, tickers = tickers, marketdata = price_matrix,
                marketfactor = gm_ema, sim_parameters = sim_params,
                lambda = 0.0, Δt = Δt, epsilon = ε
            ));

            # Build trigger rules (same for all grid points)
            rules = build(MyTriggerRules, (
                max_drawdown = 0.15, max_turnover = 0.50,
                rebalance_schedule = ones(Int, n_trading_days)
            ));

            # Run the engine and compute the wealth series
            results = run_rebalancing_engine(ctx, rules, λ_new;
                offset=offset, allocator=:cobb_douglas);
            wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

            # --- Step 4: Compute Sharpe ratio for this (G, ε) pair ---
            returns = diff(wealth) ./ wealth[1:end-1];       # daily simple returns
            total_ret = (wealth[end] / wealth[1] - 1.0);     # cumulative return (fraction)
            vol = std(returns) * sqrt(252);                   # annualized volatility
            sharpe_grid[i, j] = vol > 0 ? total_ret / vol : 0.0;
            wealth_grid[i, j] = wealth[end];                  # terminal wealth
        end
    end

    println("Sensitivity analysis complete: $(length(G_values)) × $(length(ε_values)) grid")
end

The code below generates a heatmap of Sharpe ratio across the $(G, \epsilon)$ parameter space.

In [ ]:
let
    # --- Step 1: Define axis tick labels matching the sweep ---
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    # --- Step 2: Plot the Sharpe ratio heatmap ---
    heatmap(string.(ε_values), string.(G_values), sharpe_grid,
        xlabel="ε (min share floor)", ylabel="G (lambda gain)",
        title="Sharpe Ratio: Sensitivity to G and ε",
        color=:viridis, size=(700, 500),
        clim=(minimum(sharpe_grid), maximum(sharpe_grid)))
end

The code below generates a heatmap of final wealth across the same $(G, \epsilon)$ parameter space.

In [ ]:
let
    # --- Step 1: Define axis tick labels matching the sweep ---
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    # --- Step 2: Plot the final wealth heatmap ---
    heatmap(string.(ε_values), string.(G_values), wealth_grid,
        xlabel="ε (min share floor)", ylabel="G (lambda gain)",
        title="Final Wealth (\$): Sensitivity to G and ε",
        color=:viridis, size=(700, 500))
end

The code below identifies the best and worst parameter combinations from the `sharpe_grid` and `wealth_grid` matrices.

In [ ]:
let
    # --- Step 1: Recreate parameter grids for indexing ---
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    # --- Step 2: Find the (G, ε) pair with the highest and lowest Sharpe ---
    best_idx = argmax(sharpe_grid);    # CartesianIndex of best Sharpe
    worst_idx = argmin(sharpe_grid);   # CartesianIndex of worst Sharpe

    # --- Step 3: Print results ---
    println("Best Sharpe:  G=$(G_values[best_idx[1]]), ε=$(ε_values[best_idx[2]]) → Sharpe=$(round(sharpe_grid[best_idx], digits=3)), Wealth=\$$(round(wealth_grid[best_idx], digits=2))")
    println("Worst Sharpe: G=$(G_values[worst_idx[1]]), ε=$(ε_values[worst_idx[2]]) → Sharpe=$(round(sharpe_grid[worst_idx], digits=3)), Wealth=\$$(round(wealth_grid[worst_idx], digits=2))")
end

___
## Summary
This example demonstrated how to run the Cobb-Douglas rebalancing engine with production-style trigger rules, compare it against baseline strategies via a scorecard, and map the parameter sensitivity landscape. The engine results and scorecard are saved for Session 3, where we stress-test across HMM-generated regime-switching paths and introduce a bandit challenger to compete with the utility-based allocator.

### Key Takeaways
* **Drawdown limits act as circuit breakers:** Tighter drawdown thresholds cause the engine to de-risk to cash earlier, protecting capital at the cost of missing recoveries. Looser thresholds allow more volatility but capture more upside.
* **The scorecard quantifies the adaptability trade-off:** The Cobb-Douglas engine adapts to market conditions by adjusting allocations, but this comes at the cost of higher turnover. The scorecard measures return, volatility, Sharpe ratio, and maximum drawdown to compare strategies.
* **Sensitivity analysis reveals a parameter sweet spot:** The lambda gain and epsilon parameters control engine responsiveness and diversification, respectively. Sweeping these parameters exposes regions where the engine is unresponsive, over-reactive, or well-tuned.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.